# DTDLINK Stage 4 — EDA and leakage-safe splitting

## tl;dr

- The Stage 3 table contains **641,113 unique URLs** and the fixed **36-feature schema** passed its contract checks.
- URLs were grouped by registrable domain; only 14 rows needed a URL-hash fallback because no domain was available.
- The reproducible grouped split contains **384,832 train**, **129,332 validation**, and **126,949 test** rows.
- There is **zero registrable-domain group overlap** between the three splits.
- Model-directed EDA below uses **training rows only**. The test set is locked for one final Stage 7 evaluation.
- A major data-source warning remains: explicit-scheme formatting is strongly associated with the original source labels, so model results must not be interpreted as proof that a scheme itself causes danger.

## Context & Methods

**Question.** Is the feature table trustworthy, and can it be divided into a fair train/validation/test experiment without the same domain appearing in multiple splits?

**Method.** Validate the Stage 3 schema and fingerprint; create a label-independent group key; form ten stratified grouped folds; choose the deterministic fold allocation closest to 60/20/20 while preserving class balance; verify zero overlap; then run distributions, class comparisons, and correlations on training rows only.

### Key Assumptions

- One cleaned URL is the row grain.
- `registrable_domain` is the leakage-control group where available.
- A missing domain receives a unique SHA-256 URL fallback rather than joining one artificial empty group.
- Target labels may balance the partition, but they never enter the group key or numeric feature calculations.
- Validation and test feature distributions are not used for feature selection or transformation fitting.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
from IPython.display import Image, display

current = Path.cwd().resolve()
PROJECT_ROOT = current.parent if current.name == 'notebooks' else current
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / 'src'))
os.environ.setdefault('MPLCONFIGDIR', '/tmp/dtdlink-mpl')

from dtdlink_ml.data_splitting import run_stage4_pipeline

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 140)
PROJECT_ROOT

## Data

Run the complete deterministic Stage 4 pipeline. Re-running this cell must preserve the split manifest hash.

In [ ]:
audit = run_stage4_pipeline()
print('Validation passed:', audit['validation']['passed'])
print('Manifest SHA-256:', audit['manifest']['sha256'])
print('Group overlap:', audit['validation']['pairwise_group_overlap'])

In [ ]:
profile = json.loads(Path('reports/eda/pre_split_profile.json').read_text(encoding='utf-8'))
grouping = profile['grouping_feasibility']
pd.Series({
    'Rows': profile['dataset']['rows'],
    'Unique URLs': profile['dataset']['unique_urls'],
    'Registrable domains': grouping['registrable_domain_count'],
    'Rows without domain': grouping['rows_without_registrable_domain'],
    'Singleton groups': grouping['singleton_groups'],
    'Repeated groups': grouping['repeated_groups'],
    'Largest group rows': grouping['largest_group_rows'],
}, name='Pre-split profile').to_frame()

In [ ]:
source_quality = pd.read_csv('reports/tables/source_quality_profile.csv')
source_quality.style.format({
    column: '{:.2%}' for column in source_quality.columns if column.endswith('_rate')
})

The table above is a **data-quality warning**, not a feature-selection result. For example, all defacement rows have an explicit scheme while only a small minority of benign rows do. A model may learn collection/source formatting, so later evaluation must preserve source-label subgroup checks and cautious interpretation.

## Results — grouped split

Only counts and label balance are shown across validation/test. No validation/test feature distribution is used to make modelling decisions.

In [ ]:
split_rows = []
for split_name, values in audit['validation']['splits'].items():
    split_rows.append({
        'split': split_name,
        'rows': values['rows'],
        'row_share': values['row_share'],
        'groups': values['groups'],
        'benign': values['target_counts']['0'],
        'dangerous': values['target_counts']['1'],
        'dangerous_rate': values['dangerous_rate'],
    })
split_table = pd.DataFrame(split_rows)
split_table.style.format({'row_share': '{:.2%}', 'dangerous_rate': '{:.2%}'})

In [ ]:
display(Image(filename='reports/figures/class_balance_train.png', width=760))
display(Image(filename='reports/figures/domain_concentration.png', width=900))

## Results — training-only EDA

The next outputs may influence later modelling, so they are calculated only from the training split. Histograms and boxplots use a deterministic 50,000-row training sample for readability; their clipping limits and all tabular statistics come from the full training split.

In [ ]:
comparison = pd.read_csv('reports/tables/train_feature_class_comparison.csv')
comparison.head(12).style.format({
    column: '{:.4f}' for column in comparison.columns if column != 'feature'
})

In [ ]:
display(Image(filename='reports/figures/numeric_feature_distributions.png', width=1100))
display(Image(filename='reports/figures/feature_class_boxplots.png', width=1100))
display(Image(filename='reports/figures/binary_feature_rates.png', width=950))

In [ ]:
correlation_pairs = pd.read_csv('reports/tables/high_correlation_pairs.csv')
display(correlation_pairs)
display(Image(filename='reports/figures/train_feature_correlation.png', width=1050))

## Takeaways

1. The feature table passed its schema, uniqueness, completeness, finite-value, non-negative, and binary-feature checks.
2. Domain repetition is material: 40,609 groups contain more than one URL and the largest contains 13,422 rows. Random row splitting would therefore create a serious memorisation risk.
3. The grouped split is close to 60/20/20 and has zero group overlap. Its manifest hash is the test-set lock identity.
4. `has_scheme` and `parse_issue_count` are almost perfectly inversely correlated because a recoverable missing scheme contributes a parser issue. `has_explicit_port` and `has_nonstandard_port` are also almost duplicates in this dataset. These are warnings for Stage 5 validation, not automatic deletion rules.
5. `brand_in_userinfo` and `has_confusable_unicode` are constant zero in training. They remain in the locked Stage 3 schema for now; Stage 5 must judge them through validation and product-value considerations rather than silently changing the extractor.
6. Scaling and any optional TLD-frequency lookup must be fitted on training rows only. The test set remains untouched until Stage 7.